# Predictions and flood-map scores: adding optical images

Runs the optical + radar + elevation model (best checkpoint on seg_val) over the error-spotter
folds and the 12 holdout events, and compares it on the holdout with the radar threshold and
with the radar + elevation model, whose saved predictions are read, not recomputed.
For every patch it saves the class probabilities and the true mask, which the error spotter
is trained and tested on, plus a confusion matrix per patch for the flood-map scores.

The holdout is only scored here; nothing in this notebook or later is tuned on it.

Before running: put the `spotter_fold0` to `spotter_fold4`, `holdout_main` and `holdout_storm2024`
shard folders into `MyDrive/flood_triage/shards/`, next to `seg_train` and `seg_val`.
If Colab disconnects, run all cells again: work that is already saved is skipped.

## Machine

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import os
print("CPU cores:", os.cpu_count())

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
CPU cores: 12


## Settings

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/flood_triage"
RUNS = {                                   # run name: the inputs that model was trained on
    "terramind_tiny_s1dem_t2": ["S1RTC", "DEM"],
    "terramind_tiny_s1s2dem_t2": ["S2L2A", "S1RTC", "DEM"],
}
GROUPS = [f"spotter_fold{k}" for k in range(5)] + ["holdout_main", "holdout_storm2024"]
HOLDOUT = ["holdout_main", "holdout_storm2024"]
LOCAL_PRED = "/content/predictions"
BATCH_SIZE = 32

## Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

for run in RUNS:
    ckpt = f"{DRIVE_ROOT}/runs/{run}/checkpoints/best_val_loss.ckpt"
    print(run, "checkpoint", "found" if os.path.exists(ckpt) else "not found (skipped below)")
for g in GROUPS:
    folder = f"{DRIVE_ROOT}/shards/{g}"
    n = len([f for f in os.listdir(folder) if f.endswith(".tar")]) if os.path.isdir(folder) else 0
    print(g, n, "shards")
    assert n > 0, f"no shards in {folder}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
terramind_tiny_s1dem_t2 checkpoint found
terramind_tiny_s1s2dem_t2 checkpoint found
spotter_fold0 9 shards
spotter_fold1 10 shards
spotter_fold2 14 shards
spotter_fold3 14 shards
spotter_fold4 13 shards
holdout_main 4 shards
holdout_storm2024 1 shards


## Install
Same versions as for training.

In [ ]:
%%capture install_log
!pip install terratorch==1.2.1 zarr==2.18.0 numcodecs==0.15.1 xarray==2025.10.1 torchgeo==0.9.0 webdataset==1.0.2
!test -d /content/ImpactMesh || git clone -q https://github.com/IBM/ImpactMesh.git /content/ImpactMesh
!git -C /content/ImpactMesh checkout -q 451038f21264256590eee0c73bec958897895213
!pip install -e /content/ImpactMesh --no-deps

In [ ]:
import subprocess, sys

check = "import torch, importlib.metadata as m; import webdataset, impactmesh; print(torch.__version__, torch.cuda.is_available(), m.version('terratorch'))"
r = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])

2.11.0+cu128 True 1.2.1
 


In [ ]:
# stop here if the install changed packages this session has already loaded
import importlib.metadata, importlib.util, numpy
if numpy.__version__ != importlib.metadata.version("numpy") or importlib.util.find_spec("impactmesh") is None:
    raise RuntimeError("Packages were just installed or updated: click Restart in the notebook toolbar, then Run All.")
print("environment ready")

environment ready


## Project code
Shard reader and prediction code.

In [ ]:
!mkdir -p /content/code

In [ ]:
%%writefile /content/code/triage_data.py
"""Reads the training shards.

Each sample holds s2l2a, s1rtc, dem and mask arrays (.npy, Blosc-compressed) and a meta.json.
Shapes: S2L2A (2, 12, 256, 256), S1RTC (2, 2, 256, 256), DEM (1, 256, 256), mask (256, 256);
time 0 is pre-event and time 1 is the event.
"""
import io
import json
import os
from functools import partial

import numpy as np
from numcodecs import Blosc

CODEC = Blosc(cname="zstd", clevel=5, shuffle=Blosc.SHUFFLE)
FILES = {"S2L2A": "s2l2a.npy.blosc", "S1RTC": "s1rtc.npy.blosc", "DEM": "dem.npy.blosc"}


def load_array(data):
    return np.load(io.BytesIO(CODEC.decode(data)))


def decode_sample(sample, modalities=("S2L2A", "S1RTC", "DEM")):
    """Raw shard sample -> the dict the ImpactMesh transforms expect. Only the given modalities
    are decompressed, which saves CPU time when training without S2."""
    out = {"__key__": sample["__key__"], "meta": json.loads(sample["meta.json"])}
    for mod in modalities:
        out[mod] = load_array(sample[FILES[mod]])
    out["mask"] = load_array(sample["mask.npy.blosc"]).astype(np.int64)
    return out


IMPORT_ERROR = None   # set if the datamodule could not be defined

try:  # the datamodule needs torch, terratorch and impactmesh
    if os.environ.get("TRIAGE_NO_TORCH") == "1":
        raise ImportError("TRIAGE_NO_TORCH is set")
    import webdataset as wds
    from torch.utils.data._utils.collate import default_collate
    from impactmesh.wds_impactmesh_datamodule import WdsImpactMeshDataModule
    from impactmesh.wds_impactmesh_dataset import make_transform

    class TriageWdsDataModule(WdsImpactMeshDataModule):
        """The ImpactMesh WebDataset datamodule, reading our shards. URLs can be one brace
        pattern or a list of them."""

        def _pipeline(self, urls, transform, shuffle):
            if isinstance(urls, (list, tuple)):
                urls = [u for pat in urls for u in wds.shardlists.expand_urls(pat)]
            steps = [
                wds.ResampledShards(urls) if shuffle else wds.SimpleShardList(urls),
                wds.split_by_node,
                wds.split_by_worker,
                wds.tarfile_to_samples(),
                wds.map(partial(decode_sample, modalities=tuple(self.modalities))),
            ]
            if shuffle:
                steps.append(wds.shuffle(self.shuffle_buffer))
            steps.append(wds.map(make_transform(
                self.modalities, self.timesteps, self.no_data_value,
                self.no_data_replace, transform, self.concat_bands)))
            steps.append(wds.batched(self.batch_size, collation_fn=default_collate, partial=not shuffle))
            return wds.DataPipeline(*steps)
except ImportError as e:
    IMPORT_ERROR = e

Overwriting /content/code/triage_data.py


In [ ]:
%%writefile /content/code/predict_shards.py
"""Runs the trained model over shard folders. For each patch it saves the class probabilities
(0-255, so they fit in uint8) and the true mask, and records a 3x3 confusion matrix
(rows = true class, columns = predicted class; class 0 no water, 1 permanent water, 2 flood).
"""
import io
import os

import numpy as np
import pandas as pd
import torch
import webdataset as wds

import triage_data
from triage_data import CODEC

if triage_data.IMPORT_ERROR is not None:
    raise ImportError("the shard datamodule needs terratorch and impactmesh; if they were just "
                      "installed, restart the session") from triage_data.IMPORT_ERROR
TriageWdsDataModule = triage_data.TriageWdsDataModule

CLASSES = ["no_water", "permanent", "flood"]


def pack(arr):
    buf = io.BytesIO()
    np.save(buf, arr)
    return CODEC.encode(buf.getvalue())


def load_model(ckpt, device):
    from terratorch.tasks import SemanticSegmentationTask
    task = SemanticSegmentationTask.load_from_checkpoint(ckpt, map_location=device)
    return task.eval().to(device)


def make_datamodule(modalities, batch_size, workers):
    # same settings as the training config; only the shard reader and normalisation are used
    return TriageWdsDataModule(train_urls="", val_urls="", test_urls={}, train_epoch_size=1,
                               batch_size=batch_size, num_workers=workers, modalities=list(modalities),
                               timesteps=[0, 1], disaster="flood", no_data_replace=0)


def confusion(pred, mask, n=3):
    ok = mask >= 0                                   # -1 = ignore
    return np.bincount(mask[ok] * n + pred[ok], minlength=n * n).reshape(n, n)


@torch.no_grad()
def predict_group(task, dm, urls, out_dir, group, device):
    """Predicts every patch in `urls`, writes pred-*.tar to out_dir, returns one row per patch."""
    os.makedirs(out_dir, exist_ok=True)
    pattern = os.path.join(out_dir, "pred-%06d.tar").replace("\\", "/")
    if os.name == "nt":
        pattern = "file:" + pattern
    use_amp = device.type == "cuda"
    dtype = torch.bfloat16 if use_amp and torch.cuda.get_device_capability(device)[0] >= 8 else torch.float16
    loader = dm._loader(dm._pipeline(urls, dm.test_transform, shuffle=False))
    rows = []
    with wds.ShardWriter(pattern, maxsize=200e6, verbose=0) as sink:
        for batch in loader:
            ids = batch["filename"]
            masks = batch["mask"].numpy()
            image = {m: t.to(device) for m, t in batch["image"].items()}
            image = dm.aug({"image": image})["image"]          # same normalisation as in training
            with torch.autocast(device.type, dtype=dtype, enabled=use_amp):
                logits = task(image).output
            prob = torch.softmax(logits.float(), dim=1).cpu().numpy()
            for pid, p, m in zip(ids, prob, masks):
                cm = confusion(p.argmax(0), m)
                rows.append({"patch_id": pid, "group": group, "event": pid.split("_")[0],
                             **{f"cm_{i}{j}": int(cm[i, j]) for i in range(3) for j in range(3)}})
                sink.write({"__key__": pid,
                            "prob.npy.blosc": pack(np.round(p * 255).astype(np.uint8)),
                            "mask.npy.blosc": pack(m.astype(np.int8))})
    return rows


def _samples(path, wanted):
    """Yields {name: bytes} per patch from a shard, keeping only the wanted entries."""
    import tarfile
    with tarfile.open(path) as tar:
        key, item = None, {}
        for m in tar:
            k, ext = m.name.split(".", 1)
            if key is not None and k != key:
                yield key, item
                item = {}
            key = k
            if ext in wanted:
                item[ext] = tar.extractfile(m).read()
        if item:
            yield key, item


def vv_baseline(shard_paths, threshold=-14.6):
    """Simple baseline: water where VV after the event is below `threshold` dB (chosen earlier on
    non-holdout patches). Per patch: tp = flood pixels marked water, fp = no-water pixels marked water,
    fn = flood pixels not marked. Permanent water is not counted against it (a threshold cannot tell a
    river from a flood). Patches without a radar image after the event are skipped."""
    import json
    from triage_data import load_array
    rows = []
    for path in shard_paths:
        for pid, item in _samples(path, {"s1rtc.npy.blosc", "mask.npy.blosc", "meta.json"}):
            meta = json.loads(item["meta.json"])
            if meta.get("s1_data_mask") and not meta["s1_data_mask"][1]:
                continue
            vv = load_array(item["s1rtc.npy.blosc"]).astype(np.float32)[1, 0]
            mask = load_array(item["mask.npy.blosc"])
            ok = np.isfinite(vv) & (mask >= 0)
            water = (vv < threshold) & ok
            rows.append({"patch_id": pid, "event": pid.split("_")[0],
                         "tp": int((water & (mask == 2)).sum()), "fp": int((water & (mask == 0)).sum()),
                         "fn": int((~water & ok & (mask == 2)).sum())})
    return rows


def water_rule_counts(patches):
    """The model scored with the baseline's rule: its flood and permanent water both count as water."""
    return pd.DataFrame({"patch_id": patches.patch_id, "event": patches.event,
                         "tp": patches.cm_21 + patches.cm_22, "fp": patches.cm_01 + patches.cm_02,
                         "fn": patches.cm_20})


def iou_from_counts(df):
    g = df.groupby("event")[["tp", "fp", "fn"]].sum()
    g["flood_px"] = g.tp + g.fn
    g["iou"] = g.tp / (g.tp + g.fp + g.fn)
    return g


def iou_per_event(patches, cls=2):
    """IoU of one class per event, from summed confusion matrices. NaN when the event has
    neither true nor predicted pixels of that class."""
    cols = [f"cm_{i}{j}" for i in range(3) for j in range(3)]
    g = patches.groupby(["group", "event"])[cols].sum()
    tp = g[f"cm_{cls}{cls}"]
    fp = sum(g[f"cm_{i}{cls}"] for i in range(3) if i != cls)
    fn = sum(g[f"cm_{cls}{j}"] for j in range(3) if j != cls)
    out = pd.DataFrame({"patches": patches.groupby(["group", "event"]).size(),
                        "true_px": tp + fn, "pred_px": tp + fp, "iou": tp / (tp + fp + fn)})
    return out.reset_index()

Overwriting /content/code/predict_shards.py


## Predict

For each model with a checkpoint: shards are read straight from Drive (a single pass, so copying
them first would not save time). Each group's predictions are written locally, then copied to the
run's `predictions` folder on Drive.

In [ ]:
import glob, os, sys, time
import pandas as pd
import torch

sys.path[:0] = ["/content/code", "/content/ImpactMesh"]   # our code and the ImpactMesh repo
for name in ["triage_data", "predict_shards"]:   # load the files written above, not older copies
    sys.modules.pop(name, None)                     # imported earlier in this session
import predict_shards as ps

device = torch.device("cuda")
for run, modalities in RUNS.items():
    run_dir = f"{DRIVE_ROOT}/runs/{run}"
    ckpt, pred_dir = f"{run_dir}/checkpoints/best_val_loss.ckpt", f"{run_dir}/predictions"
    todo = [g for g in GROUPS if not os.path.exists(f"{pred_dir}/{g}_patches.csv")]
    if not os.path.exists(ckpt) or not todo:
        print(run, "no checkpoint yet" if not os.path.exists(ckpt) else "all groups done")
        continue
    task = ps.load_model(ckpt, device)
    dm = ps.make_datamodule(modalities, BATCH_SIZE, workers=min(4, os.cpu_count() or 2))
    os.makedirs(pred_dir, exist_ok=True)
    for g in todo:
        t0 = time.time()
        urls = sorted(glob.glob(f"{DRIVE_ROOT}/shards/{g}/shard-*.tar"))
        local = f"{LOCAL_PRED}/{run}/{g}"
        rows = ps.predict_group(task, dm, urls, local, g, device)
        !mkdir -p "{pred_dir}/{g}" && rsync -a "{local}/" "{pred_dir}/{g}/"
        pd.DataFrame(rows).to_csv(f"{pred_dir}/{g}_patches.csv", index=False)   # written last: a half-done group is redone
        print(f"{run} {g}: {len(rows)} patches in {time.time() - t0:.0f} s")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


terramind_tiny_s1dem_t2 all groups done


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


TerraMind_v1_tiny.pt: reconstructing file:   0%|          |  0.00B /  212MB            

TerraMind_v1_tiny.pt: downloading bytes:           |  0.00B            

terramind_tiny_s1s2dem_t2 spotter_fold0: 1091 patches in 50 s
terramind_tiny_s1s2dem_t2 spotter_fold1: 1196 patches in 47 s
terramind_tiny_s1s2dem_t2 spotter_fold2: 1649 patches in 61 s
terramind_tiny_s1s2dem_t2 spotter_fold3: 1720 patches in 67 s
terramind_tiny_s1s2dem_t2 spotter_fold4: 1585 patches in 63 s
terramind_tiny_s1s2dem_t2 holdout_main: 492 patches in 16 s
terramind_tiny_s1s2dem_t2 holdout_storm2024: 70 patches in 11 s


## Radar threshold baseline (holdout)

Water where VV after the event is below -14.6 dB; the threshold was chosen earlier on patches outside
the holdout. A threshold cannot tell a river from a flood, so its score counts flood pixels marked water
as hits and no-water pixels marked water as false alarms, and does not count permanent water against it.

In [ ]:
base_file = f"{DRIVE_ROOT}/runs/vv_baseline_holdout.csv"
if not os.path.exists(base_file):
    rows = ps.vv_baseline([u for g in HOLDOUT for u in sorted(glob.glob(f"{DRIVE_ROOT}/shards/{g}/shard-*.tar"))])
    pd.DataFrame(rows).to_csv(base_file, index=False)
baseline = ps.iou_from_counts(pd.read_csv(base_file))
baseline.round(3)

,tp,fp,fn,flood_px,iou
event,,,,,
EMSR251,3448,49229,1793,5241,0.063
EMSR321,339033,40719,210138,549171,0.575
EMSR358,6068,69616,1781,7849,0.078
EMSR556,3666,10731,38395,42061,0.069
EMSR757,2135,800,2697,4832,0.379
EMSR763,134964,270559,17930,152894,0.319
EMSR766,0,2600,0,0,0.000
EMSR841,65,1690,29,94,0.036
EMSR844,7124,16167,2509,9633,0.276


## Flood-map scores on the holdout

IoU per event, then the mean over events with flood. Two ways to score a model:

- **flood IoU**: only the model's flood class counts; permanent water predicted as flood is an error.
- **baseline rule**: the model's flood and permanent water both count as water, exactly as for the
  threshold, so this column is the fair comparison with it.

Events without flood are left out of the means (IoU undefined). Intervals resample events.

In [ ]:
storm = {"EMSR757", "EMSR763", "EMSR766"}
per_event = {"radar threshold (baseline rule)": baseline.iou}
for run in RUNS:
    pred_dir = f"{DRIVE_ROOT}/runs/{run}/predictions"
    if not all(os.path.exists(f"{pred_dir}/{g}_patches.csv") for g in GROUPS):
        continue
    patches = pd.concat(pd.read_csv(f"{pred_dir}/{g}_patches.csv") for g in GROUPS)
    patches.to_csv(f"{pred_dir}/all_patches.csv", index=False)
    hold = patches[patches.group.isin(HOLDOUT)]
    per_event[f"{run} (baseline rule)"] = ps.iou_from_counts(ps.water_rule_counts(hold)).iou
    per_event[f"{run} (flood IoU)"] = ps.iou_per_event(hold, cls=2).set_index("event").iou
    ps.iou_per_event(patches, cls=2).to_csv(f"{pred_dir}/flood_iou_per_event.csv", index=False)

per_event = pd.DataFrame(per_event)[baseline.flood_px > 0]          # events with flood
per_event.to_csv(f"{DRIVE_ROOT}/runs/holdout_iou_per_event.csv")
summary = pd.DataFrame({"all events with flood": per_event.mean(),
                        "without Sept 2024 storm": per_event[~per_event.index.isin(storm)].mean()})
print(f"{len(per_event)} holdout events with flood ({len(per_event[~per_event.index.isin(storm)])} without the storm)\n")
print(summary.round(3))
per_event.round(3)

11 holdout events with flood (9 without the storm)

                                           all events with flood  \
radar threshold (baseline rule)                            0.258   
terramind_tiny_s1dem_t2 (baseline rule)                    0.377   
terramind_tiny_s1dem_t2 (flood IoU)                        0.378   
terramind_tiny_s1s2dem_t2 (baseline rule)                  0.408   
terramind_tiny_s1s2dem_t2 (flood IoU)                      0.412   

                                           without Sept 2024 storm  
radar threshold (baseline rule)                              0.238  
terramind_tiny_s1dem_t2 (baseline rule)                      0.313  
terramind_tiny_s1dem_t2 (flood IoU)                          0.317  
terramind_tiny_s1s2dem_t2 (baseline rule)                    0.366  
terramind_tiny_s1s2dem_t2 (flood IoU)                        0.371  


,radar threshold (baseline rule),terramind_tiny_s1dem_t2 (baseline rule),terramind_tiny_s1dem_t2 (flood IoU),terramind_tiny_s1s2dem_t2 (baseline rule),terramind_tiny_s1s2dem_t2 (flood IoU)
event,,,,,
EMSR251,0.063,0.066,0.022,0.070,0.300
EMSR321,0.575,0.626,0.572,0.633,0.590
EMSR358,0.078,0.376,0.506,0.431,0.419
EMSR556,0.069,0.268,0.276,0.286,0.290
EMSR757,0.379,0.637,0.636,0.621,0.624
EMSR763,0.319,0.697,0.663,0.579,0.570
EMSR841,0.036,0.025,0.188,0.000,0.000
EMSR844,0.276,0.328,0.263,0.326,0.306
EMSR845,0.369,0.432,0.414,0.397,0.385


Average flood-map score (IoU, 1 = perfect): optical model 0.41, radar model 0.38, threshold 0.26. Optical fixes the two floods where radar failed (EMSR867, EMSR251), but is a bit worse on two others.

In [ ]:
import numpy as np

def paired_bootstrap(a, b, n=20000, seed=0):
    d = (a - b).to_numpy()
    idx = np.random.default_rng(seed).integers(0, len(d), (n, len(d)))
    lo, hi = np.percentile(d[idx].mean(1), [2.5, 97.5])
    return d.mean(), lo, hi

def compare(a, b):
    for name, sel in [("all", per_event.index), ("without storm", per_event.index[~per_event.index.isin(storm)])]:
        d, lo, hi = paired_bootstrap(per_event.loc[sel, a], per_event.loc[sel, b])
        wins = (per_event.loc[sel, a] > per_event.loc[sel, b]).sum()
        print(f"  {name:14s} {d:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  better in {wins}/{len(sel)} events")

runs_done = [r for r in RUNS if f"{r} (baseline rule)" in per_event]
for run in runs_done:
    print(f"{run} minus radar threshold (both with the baseline rule):")
    compare(f"{run} (baseline rule)", "radar threshold (baseline rule)")
if len(runs_done) == 2:
    a, b = runs_done[1], runs_done[0]
    print(f"\n{a} minus {b} (flood IoU):")
    compare(f"{a} (flood IoU)", f"{b} (flood IoU)")

terramind_tiny_s1dem_t2 minus radar threshold (both with the baseline rule):
  all            +0.120  95% CI [+0.020, +0.215]  better in 9/11 events
  without storm  +0.075  95% CI [-0.023, +0.169]  better in 7/9 events
terramind_tiny_s1s2dem_t2 minus radar threshold (both with the baseline rule):
  all            +0.150  95% CI [+0.076, +0.224]  better in 10/11 events
  without storm  +0.128  95% CI [+0.047, +0.212]  better in 8/9 events

terramind_tiny_s1s2dem_t2 minus terramind_tiny_s1dem_t2 (flood IoU):
  all            +0.034  95% CI [-0.053, +0.142]  better in 6/11 events
  without storm  +0.053  95% CI [-0.052, +0.177]  better in 6/9 events


Both models clearly beat the threshold; the confidence range is entirely above zero. Optical vs radar: only +0.03, and the range includes zero, so with 11 floods we can't say optical is better. Optical helps when the sky is clear; Nepal was cloudy for weeks, so radar is what matters there.